In [1]:
## Step 1 - Imports ##

import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
## Step 2 - Load the data ##

# Load well datasets
well1 = pd.read_csv("1_2021002_.csv")
well4 = pd.read_csv("4_20211022_.csv")
well7 = pd.read_csv("7_2021022_.csv")

In [6]:
## Step 3 - Clean and combine ##

# Fix column issues
well4 = well4.rename(columns={"litholody": "lithology"})
well7 = well7.rename(columns={"litholody": "lithology"})

# Add well labels
well1["well"] = "well1"
well4["well"] = "well4"
well7["well"] = "well7"

# Combine
all_data = pd.concat([well1, well4, well7], axis=0)

# Keep relevant columns (no depth again)
columns = ["SP", "GR", "LLD", "LLS", "DEN", "AC", "lithology", "well"]
all_data = all_data[columns]

# Remove missing lithology
all_data = all_data.dropna(subset=["lithology"])

In [7]:
## Step 4 - Define X and y ##

X = all_data[["SP", "GR", "LLD", "LLS", "DEN"]]
y = all_data["AC"]

In [8]:
## Step 5 - Add constant ##

X_sm = sm.add_constant(X)

In [9]:
## Step 6 - Fit model - OLS ##

model = sm.OLS(y, X_sm)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                     AC   R-squared:                       0.544
Model:                            OLS   Adj. R-squared:                  0.544
Method:                 Least Squares   F-statistic:                     6354.
Date:                Fri, 26 Jun 2026   Prob (F-statistic):               0.00
Time:                        02:26:16   Log-Likelihood:                -99092.
No. Observations:               26604   AIC:                         1.982e+05
Df Residuals:                   26598   BIC:                         1.982e+05
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        301.4110      1.421    212.166      0.0

In [11]:
## Step 7 - VIF ##

def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns

    vif_data["VIF"] = [
        1 / (1 - sm.OLS(X[col], sm.add_constant(X.drop(columns=[col]))).fit().rsquared)
        for col in X.columns
    ]

    return vif_data

vif = calculate_vif(X)
print(vif)

  feature        VIF
0      SP   1.406662
1      GR   1.796021
2     LLD  13.280053
3     LLS  13.115050
4     DEN   1.348462


In [25]:
## Step 8 - Interpretation  ##

#A multiple linear regression model was developed to predict the acoustic log (AC) using other well log measurements (SP, GR, LLD, LLS, and DEN). The Variance Inflation Factor (VIF) analysis was conducted to evaluate multicollinearity among the input variables. The results show that SP, GR, and DEN have low VIF values (approximately 1.3–1.8), indicating that these variables are not strongly correlated with each other and provide independent information to the model. However, LLD and LLS exhibit high VIF values (greater than 13), suggesting strong multicollinearity between these resistivity measurements. This indicates redundancy, as both variables capture similar subsurface properties. High multicollinearity can negatively affect the stability and interpretability of regression coefficients. Overall, the model demonstrates that well log measurements can be used to reasonably predict the acoustic log, but careful feature selection is necessary to reduce redundancy and improve model robustness. In particular, one of the highly correlated variables -LLD or LLS- could be removed to improve the model.

Optimization terminated successfully.
         Current function value: 0.351021
         Iterations 8


In [29]:
## Step 9 - Print Full Summary ##

print(result.summary())

                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                20050
Model:                          Logit   Df Residuals:                    20043
Method:                           MLE   Df Model:                            6
Date:                Fri, 26 Jun 2026   Pseudo R-squ.:                  0.3502
Time:                        02:14:23   Log-Likelihood:                -7038.0
converged:                       True   LL-Null:                       -10831.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const        -14.4911      0.919    -15.764      0.000     -16.293     -12.689
SP            -0.0419      0.001    -31.381      0.000      -0.045      -0.039
GR             0.0189      0.003      6.863      0.0

In [30]:
## Step 10 - Predictions ##

y_pred_prob = result.predict(X_test_sm)

# Convert probabilities → 0 or 1
y_pred = (y_pred_prob >= 0.5).astype(int)

In [31]:
## Step 11 - Accuracy ##

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.878089716203845


In [32]:
## Step 12 - Confusion Matrix ##

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[1576  700]
 [  99 4179]]


In [15]:
## Step 13 - Remove high VIF LLS values ##
X_new1 = all_data[["SP", "GR", "LLD", "DEN"]]
vif_new1 = calculate_vif(X_new1)
print(vif_new1)

  feature       VIF
0      SP  1.397203
1      GR  1.792638
2     LLD  1.074930
3     DEN  1.257613


In [16]:
## Step 14 - Remove high VIF LLD and LLS values ##
X_new2 = all_data[["SP", "GR", "DEN"]]
vif_new2 = calculate_vif(X_new2)
print(vif_new2)

  feature       VIF
0      SP  1.394462
1      GR  1.692422
2     DEN  1.254943


In [ ]:
## Interpretation

# The initial VIF analysis identified strong multicollinearity between the resistivity measurements LLD and LLS, both exhibiting VIF values greater than 13. After removing LLS, multicollinearity was significantly reduced. To further simplify the model, both LLD and LLS were removed, leaving SP, GR, and DEN as the remaining predictors. The updated VIF values for these variables were all close to 1, indicating that multicollinearity had been completely eliminated. This confirms that SP, GR, and DEN provide independent and non-redundant information for predicting the target variable. While removing LLD and LLS simplifies the model and improves interpretability, it may also slightly reduce predictive power, as some useful information from resistivity logs is discarded. Overall, the results demonstrate that careful feature selection can improve model stability and interpretability by removing redundant variables, while maintaining a strong predictive framework.
# While removing both LLD and LLS eliminates multicollinearity completely, retaining one resistivity measurement (such as LLD) provides a better balance between model simplicity and geological information.